<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.4: 函数式编程
**上一节: [高阶函数](3.3_higher-order_functions.ipynb)**<br>
**下一节: [面向对象编程](3.5_object_oriented_programming.ipynb)**

## 动机
你在之前的许多模块中看到了函数，但现在是我们自己创建并有效使用它们的时候了。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

本模块使用 Chisel 的 `FixedPoint` 类型，该类型目前位于实验包中。

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test
import chisel3.experimental._
import chisel3.internal.firrtl.KnownBinaryPoint

---
# Functional Programming in Scala
Scala 函数在模块 1 中介绍过，你在上一个模块中看到它们被大量使用。这里是对函数的复习。函数接受任意数量的输入并产生一个输出。输入通常称为函数的参数。要产生无输出，返回 `Unit` 类型。

<span style="color:blue">**示例: 自定义函数**</span><br>
下面是 Scala 中函数的一些示例。

In [ ]:
// 无输入或输出（两个版本）。
def hello1(): Unit = print("Hello!")
def hello2 = print("Hello again!")

// 数学运算：一个输入和一个输出。
def times2(x: Int): Int = 2 * x

// Inputs can have default values, and explicitly specifying the return type is optional.
// Note that we recommend specifying the return types to avoid surprises/bugs.
def timesN(x: Int, n: Int = 2) = n * x

// 调用上面列出的函数。
hello1()
hello2
times2(4)
timesN(4)         // 无需指定 n 即可使用默认值
timesN(4, 3)      // 参数顺序与定义函数时的顺序相同
timesN(n=7, x=2)  // 参数可以重新排序并显式赋值

## 函数作为对象
Scala 中的函数是一等对象。这意味着我们可以将一个函数赋给一个 `val`，并将其作为参数传递给类、对象或其他函数。

<span style="color:blue">**示例: 函数 Objects**</span><br>
下面是以函数和对象形式实现的相同函数。

In [ ]:
// 这些是普通的函数。
def plus1funct(x: Int): Int = x + 1
def times2funct(x: Int): Int = x * 2

// 这些是作为 vals 的函数。
// 第一个显式指定了返回类型。
val plus1val: Int => Int = x => x + 1
val times2val = (x: Int) => x * 2

// 调用两者看起来相同。
plus1funct(4)
plus1val(4)
plus1funct(x=4)
//plus1val(x=4) // 这不起作用

为什么要创建 `val` 而不是 `def`？使用 `val`，您现在可以将函数传递给其他函数，如下所示。您甚至可以创建自己的接受其他函数作为参数的函数。正式地说，接受或产生函数的函数被称为*高阶函数*。您在上一模块中看到过它们的使用，但现在您将创建自己的函数！

<span style="color:blue">**示例: Higher-Order Functions**</span><br>
这里我们再次展示 `map`，我们还创建一个新函数 `opN`，它接受一个函数 `op` 作为参数。

In [ ]:
// 创建我们的函数
val plus1 = (x: Int) => x + 1
val times2 = (x: Int) => x * 2

// 将它传递给 map，一个列表函数
val myList = List(1, 2, 5, 9)
val myListPlus = myList.map(plus1)
val myListTimes = myList.map(times2)

// 创建一个自定义函数，它使用递归对 X 执行 N 次操作
def opN(x: Int, n: Int, op: Int => Int): Int = {
  if (n <= 0) { x }
  else { opN(op(x), n-1, op) }
}

opN(7, 3, plus1)
opN(7, 3, times2)

<span style="color:blue">**示例: Functions vs. Objects**</span><br>
在使用无参数函数时，可能会出现一种令人困惑的情况。函数每次被调用时都会被评估，而 `val` 在实例化时被评估。

In [ ]:
import scala.util.Random

// x 和 y 都调用了 nextInt 函数，但 x 立即被评估，而 y 是一个函数
val x = Random.nextInt
def y = Random.nextInt

// x 之前已被评估，所以它是一个常量
println(s"x = $x")
println(s"x = $x")

// y 是一个函数，在每次调用时被重新评估，因此这些产生不同的结果
println(s"y = $y")
println(s"y = $y")

## 匿名函数
正如名字所暗示的，匿名函数没有名字。如果我们只使用一次函数，就无需为其创建 `val`。

<span style="color:blue">**示例: Anonymous Functions**</span><br>
以下示例演示了这一点。它们通常是有作用域的（放在大括号中而不是括号中）。

In [ ]:
val myList = List(5, 6, 7, 8)

// 使用匿名函数为列表中的每一项加一
// 参数传递给下划线变量
// 这些都做同样的事情
myList.map( (x:Int) => x + 1 )
myList.map(_ + 1)

// 一种常见情况是在匿名函数中使用 case 语句
val myAnyList = List(1, 2, "3", 4L, myList)
myAnyList.map {
  case (_:Int|_:Long) => "Number"
  case _:String => "String"
  case _ => "error"
}

<span style="color:red">**练习: Sequence Manipulation**</span><br>
您将使用的一组常见高阶函数是 `scanLeft`/`scanRight`、`reduceLeft`/`reduceRight` 和 `foldLeft`/`foldRight`。了解每个函数的工作原理以及何时使用它们非常重要。`scan`、`reduce` 和 `fold` 的默认方向是向左，但这在所有情况下都不能保证。

In [ ]:
val exList = List(1, 5, 7, 100)

// 编写一个自定义函数来相加两个数字，然后使用 reduce 查找 exList 中所有值的总和
def add(a: Int, b: Int): Int = ???
val sum = ???

// 使用匿名函数查找 exList 的总和（提示：您之前见过这个！）
val anon_sum = ???

// 使用 scan 从右到左查找 exList 的移动平均值；使结果 成为双精度值列表
def avg(a: Int, b: Double): Double = ???
val ma2 = ???

In [ ]:
assert(add(88, 88) == 176)
assert(sum == 113)

assert(anon_sum == 113)

assert(avg(100, 100.0) == 100.0)
assert(ma2 == List(8.875, 16.75, 28.5, 50.0, 0.0))

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
def add(a: Int, b: Int): Int = a + b
val sum = exList.reduce(add)

val anon\_sum = exList.reduce(\_ + \_)

def avg(a: Int, b: Double): Double = (a + b)/2.0
val ma2 = exList.scanRight(0.0)(avg)
</pre></article></div></section></div>

---
# Functional Programming in Chisel
让我们看看一些示例，了解如何在 Chisel 中创建硬件生成器时使用函数式编程。

<span style="color:blue">**示例: FIR Filter**</span><br>
首先，我们将重新审视上一示例中的 FIR 滤波器。我们不会将系数作为参数传递给类或使其可编程，而是将一个函数传递给 FIR，该函数定义了如何计算窗口系数。该函数将接受窗口长度和位宽，以生成系数的缩放列表。这里有两个示例窗口。为了避免小数，我们将系数缩放到最大和最小整数值之间。有关这些窗口的更多信息，请查看[此 Wikipedia 页面](https://en.wikipedia.org/wiki/Window_function)。

In [ ]:
// 获取一些数学函数
import scala.math.{abs, round, cos, Pi, pow}

// 简单的三角形窗口
val TriangularWindow: (Int, Int) => Seq[Int] = (length, bitwidth) => {
  val raw_coeffs = (0 until length).map( (x:Int) => 1-abs((x.toDouble-(length-1)/2.0)/((length-1)/2.0)) )
  val scaled_coeffs = raw_coeffs.map( (x: Double) => round(x * pow(2, bitwidth)).toInt)
  scaled_coeffs
}

// 汉明窗口
val HammingWindow: (Int, Int) => Seq[Int] = (length, bitwidth) => {
  val raw_coeffs = (0 until length).map( (x: Int) => 0.54 - 0.46*cos(2*Pi*x/(length-1)))
  val scaled_coeffs = raw_coeffs.map( (x: Double) => round(x * pow(2, bitwidth)).toInt)
  scaled_coeffs
}

// 测试一下！第一个参数是窗口长度，第二个参数是位宽
TriangularWindow(10, 16)
HammingWindow(10, 16)

Now we'll create a FIR filter that accepts a window 函数 as the 参数. This allows us to define new windows later on and retain the same FIR generator. It also allows us to independently size the FIR, knowing the window will be recalculated for different lengths or bitwidths. Since we are choosing the window at compile time, these coefficients are fixed.

In [ ]:
// our FIR has parameterized window length, IO bitwidth, and windowing function
class MyFir(length: Int, bitwidth: Int, window: (Int, Int) => Seq[Int]) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitwidth.W))
    val out = Output(UInt((bitwidth*2+length-1).W)) // expect bit growth, conservative but lazy
  })

  // calculate the coefficients using the provided window function, mapping to UInts
  val coeffs = window(length, bitwidth).map(_.U)
  
  // create an array holding the output of the delays
  // note: we avoid using a Vec here since we don't need dynamic indexing
  val delays = Seq.fill(length)(Wire(UInt(bitwidth.W))).scan(io.in)( (prev: UInt, next: UInt) => {
    next := RegNext(prev)
    next
  })
  
  // multiply, putting result in "mults"
  val mults = delays.zip(coeffs).map{ case(delay: UInt, coeff: UInt) => delay * coeff }
  
  // add up multiplier outputs with bit growth
  val result = mults.reduce(_+&_)

  // connect output
  io.out := result
}

visualize(() => new MyFir(7, 12, TriangularWindow))

Those last three lines could be easily combined into one. Also notice how we've handled bitwidth growth conservatively to avoid loss.

<span style="color:blue">**示例: FIR Filter 测试器**</span><br>
Let's 测试 our FIR! Previously, we provided a custom golden model. This time we'll use Breeze, a Scala library of useful linear algebra and signal processing functions, as a golden model for our FIR filter. The code below compares the Chisel 输出 with the golden model 输出, and any errors cause the 测试器 to fail.

Try uncommenting the print statment at the end just after the 期望 call. Also try changing the window from triangular to Hamming.

In [ ]:
// math imports
import scala.math.{pow, sin, Pi}
import breeze.signal.{filter, OptOverhang}
import breeze.signal.support.{CanFilter, FIRKernel1D}
import breeze.linalg.DenseVector

// test parameters
val length = 7
val bitwidth = 12 // must be less than 15, otherwise Int can't represent the data, need BigInt
val window = TriangularWindow

// test our FIR
test(new MyFir(length, bitwidth, window)) { c =>
    
    // test data
    val n = 100 // input length
    val sine_freq = 10
    val samp_freq = 100

    // sample data, scale to between 0 and 2^bitwidth
    val max_value = pow(2, bitwidth)-1
    val sine = (0 until n).map(i => (max_value/2 + max_value/2*sin(2*Pi*sine_freq/samp_freq*i)).toInt)
    //println(s"input = ${sine.toArray.deep.mkString(", ")}")

    // coefficients
    val coeffs = window(length, bitwidth)
    //println(s"coeffs = ${coeffs.toArray.deep.mkString(", ")}")

    // use breeze filter as golden model; need to reverse coefficients
    val expected = filter(
        DenseVector(sine.toArray),
        FIRKernel1D(DenseVector(coeffs.reverse.toArray), 1.0, ""),
        OptOverhang.None
    )
    expected.toArray // seems to be necessary
    //println(s"exp_out = ${expected.toArray.deep.mkString(", ")}") // this seems to be necessary

    // push data through our FIR and check the result
    c.reset.poke(true.B)
    c.clock.step(5)
    c.reset.poke(false.B)
    for (i <- 0 until n) {
        c.io.in.poke(sine(i).U)
        if (i >= length-1) { // wait for all registers to be initialized since we didn't zero-pad the data
            val expectValue = expected(i-length+1)
            //println(s"expected value is $expectValue")
            c.io.out.expect(expected(i-length+1).U)
            //println(s"cycle $i, got ${c.io.out.peek()}, expect ${expected(i-length+1)}")
        }
        c.clock.step(1)
    }
}

---
# Chisel Exercises
Complete 以下 exercises to practice writing functions, using them as arguments to 硬件 generators, and avoiding mutable data.

<span style="color:red">**练习: Neural Network Neuron**</span><br>
Our first 示例 will have you build a neuron, the building block of fully-connected layers in artificial neural networks. Neurons take inputs and a set of weights, one per 输入, and produce one 输出. The weights and inputs are multiplied and added, and the result is fed through an activation 函数. In this 练习, you will implement different activation functions and pass them as an 参数 to your neuron generator.

![Neuron](https://upload.wikimedia.org/wikipedia/commons/thumb/6/60/ArtificialNeuronModel_english.png/600px-ArtificialNeuronModel_english.png)

First, complete 以下 code to create a neuron generator. The 参数 `inputs` gives the number of inputs. The 参数 `act` is a 函数 that implements the logic of the activation 函数. We'll make the inputs and outputs 16-bit fixed point values with 8 fractional bits.

In [ ]:
class Neuron(inputs: Int, act: FixedPoint => FixedPoint) extends Module {
  val io = IO(new Bundle {
    val in      = Input(Vec(inputs, FixedPoint(16.W, 8.BP)))
    val weights = Input(Vec(inputs, FixedPoint(16.W, 8.BP)))
    val out     = Output(FixedPoint(16.W, 8.BP))
  })
  
  ???
}

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-2" />
<label for="check-2"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val mac = io.in.zip(io.weights).map{ case(a:FixedPoint, b:FixedPoint) => a*b}.reduce(_+_)
  io.out := act(mac)
</pre></article></div></section></div>

Now let's create some activation functions! We'll use a threshold of zero. Typical activation functions are the sigmoid 函数 and the rectified linear unit (ReLU).

The sigmoid we'll use is called the [logistic 函数](https://en.wikipedia.org/wiki/Logistic_function), given by 

$logistic(x) = \cfrac{1}{1+e^{-\beta x}}$

where $\beta$ is the slope factor. 然而, computing the exponential 函数 in 硬件 is quite challenging and expensive. We'll approximate this as the 步进 函数.

$步进(x) = \begin{cases}
             0  & \text{if } x \le 0 \\
             1  & \text{if } x \gt 0
       \end{cases}$

The second 函数, the ReLU, is given by a similar formula.

$relu(x) = \begin{cases}
             0  & \text{if } x \le 0 \\
             x  & \text{if } x \gt 0
       \end{cases}$

Implement these two functions below. You can specify a fixed-point 字面量 like `-3.14.F(8.BP)`. 

In [ ]:
val Step: FixedPoint => FixedPoint = ???
val ReLU: FixedPoint => FixedPoint = ???

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-3" />
<label for="check-3"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
val 步进: FixedPoint => FixedPoint = x => Mux(x <= 0.F(8.BP), 0.F(8.BP), 1.F(8.BP))
val ReLU: FixedPoint => FixedPoint = x => Mux(x <= 0.F(8.BP), 0.F(8.BP), x)
</pre></article></div></section></div>

Finally, let's create a 测试器 that checks the correctness of our Neuron. With the 步进 activation 函数, neurons may be used as logic gate approximators. Proper selection of weights and bias can perform binary functions. We'll 测试 our neuron using AND logic. Complete 以下 测试器 to check our neuron with the 步进 函数.

请注意 since the 电路 is purely combinational, the `复位(5)` and `步进(1)` calls are not necessary.

In [ ]:
// test our Neuron 
test(new Neuron(2, Step)) { c =>
    val inputs = Seq(Seq(-1, -1), Seq(-1, 1), Seq(1, -1), Seq(1, 1))

    // make this a sequence of two values
    val weights = ???

    // push data through our Neuron and check the result (AND gate)
    c.reset.poke(true.B)
    c.clock.step(5)
    c.reset.poke(false.B)
    for (i <- inputs) {
        c.io.in(0).poke(i(0).F(8.BP))
        c.io.in(1).poke(i(1).F(8.BP))
        c.io.weights(0).poke(weights(0).F(16.W, 8.BP))
        c.io.weights(1).poke(weights(1).F(16.W, 8.BP))
        c.io.out.expect((if (i(0) + i(1) > 0) 1 else 0).F(16.W, 8.BP))
        c.clock.step(1)
    }
    
}

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-4" />
<label for="check-4"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
val weights  = Seq(1.0, 1.0)
</pre></article></div></section></div>

---
# You're done!

[Return to the top.](#top)